<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day6/Parameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv('/content/cleaned_76hd.csv')

In [2]:
df.shape

(282, 39)

In [3]:
df.columns

Index(['V3', 'V4', 'V9', 'V10', 'V11', 'V12', 'V14', 'V15', 'V16', 'V18',
       'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',
       'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V37', 'V38', 'V39',
       'V40', 'V41', 'V43', 'V44', 'V51', 'V55', 'V56', 'V57', 'V58'],
      dtype='object')

In [4]:
X = df.drop(columns=['V58'])
Y = df['V58']


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=7,stratify=Y)

In [6]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((197, 38), (85, 38), (197,), (85,))

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

pg = {
    'C':[0.1,0.5,1.0],
    'gamma':[0.5,1,5]
    }
# Verbose -> Detail
gscv = GridSearchCV(SVC(random_state=7,kernel='rbf'),
                    param_grid=pg,cv=2,scoring='average_precision',verbose=2)
gscv.fit(X_train,Y_train)

Fitting 2 folds for each of 9 candidates, totalling 18 fits
[CV] END ...................................C=0.1, gamma=0.5; total time=   0.0s
[CV] END ...................................C=0.1, gamma=0.5; total time=   0.0s
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END .....................................C=0.1, gamma=1; total time=   0.0s
[CV] END .....................................C=0.1, gamma=5; total time=   0.0s
[CV] END .....................................C=0.1, gamma=5; total time=   0.0s
[CV] END ...................................C=0.5, gamma=0.5; total time=   0.0s
[CV] END ...................................C=0.5, gamma=0.5; total time=   0.0s
[CV] END .....................................C=0.5, gamma=1; total time=   0.0s
[CV] END .....................................C=0.5, gamma=1; total time=   0.0s
[CV] END .....................................C=0.5, gamma=5; total time=   0.0s
[CV] END .....................................C=0

GridSearchCV(cv=2, estimator=SVC(random_state=7),
             param_grid={'C': [0.1, 0.5, 1.0], 'gamma': [0.5, 1, 5]},
             scoring='average_precision', verbose=2)

In [9]:
gscv.best_params_

{'C': 1.0, 'gamma': 0.5}

In [10]:
svm = SVC(random_state=7,kernel='rbf',C=1,gamma=0.5)
svm.fit(X_train,Y_train)

SVC(C=1, gamma=0.5, random_state=7)

In [11]:
from sklearn.metrics import classification_report
Y_pred = svm.predict(X_test)
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80        46
           1       0.77      0.77      0.77        39

    accuracy                           0.79        85
   macro avg       0.79      0.79      0.79        85
weighted avg       0.79      0.79      0.79        85



# Parameter tuning

- RF --> max_features & n_estimators
- XGB
- CatBoost
- LightGBM

## RandomForestClassifier

In [19]:
from sklearn.ensemble import RandomForestClassifier
pg = {
    'max_features':[22,28,31],
    'max_samples':[0.6,0.8,0.9],
    'n_estimators':[50,100,200]
    }
gscv = GridSearchCV(RandomForestClassifier(random_state=7,oob_score=True),
                    param_grid=pg,cv=2,scoring='average_precision',verbose=2)
gscv.fit(X_train,Y_train)

Fitting 2 folds for each of 27 candidates, totalling 54 fits
[CV] END ..max_features=22, max_samples=0.6, n_estimators=50; total time=   0.2s
[CV] END ..max_features=22, max_samples=0.6, n_estimators=50; total time=   0.4s
[CV] END .max_features=22, max_samples=0.6, n_estimators=100; total time=   0.5s
[CV] END .max_features=22, max_samples=0.6, n_estimators=100; total time=   0.4s
[CV] END .max_features=22, max_samples=0.6, n_estimators=200; total time=   0.5s
[CV] END .max_features=22, max_samples=0.6, n_estimators=200; total time=   0.5s
[CV] END ..max_features=22, max_samples=0.8, n_estimators=50; total time=   0.1s
[CV] END ..max_features=22, max_samples=0.8, n_estimators=50; total time=   0.1s
[CV] END .max_features=22, max_samples=0.8, n_estimators=100; total time=   0.3s
[CV] END .max_features=22, max_samples=0.8, n_estimators=100; total time=   0.2s
[CV] END .max_features=22, max_samples=0.8, n_estimators=200; total time=   0.5s
[CV] END .max_features=22, max_samples=0.8, n_es

GridSearchCV(cv=2,
             estimator=RandomForestClassifier(oob_score=True, random_state=7),
             param_grid={'max_features': [22, 28, 31],
                         'max_samples': [0.6, 0.8, 0.9],
                         'n_estimators': [50, 100, 200]},
             scoring='average_precision', verbose=2)

In [20]:
gscv.best_params_

{'max_features': 22, 'max_samples': 0.6, 'n_estimators': 200}

In [21]:
print(gscv.best_score_)

0.884920957589953


In [22]:
rf = RandomForestClassifier(random_state=7,oob_score=True,max_features=22,max_samples=0.6,n_estimators=200)
rf.fit(X_train,Y_train)

RandomForestClassifier(max_features=22, max_samples=0.6, n_estimators=200,
                       oob_score=True, random_state=7)

In [23]:
Y_pred = rf.predict(X_test)
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.81      0.85      0.83        46
           1       0.81      0.77      0.79        39

    accuracy                           0.81        85
   macro avg       0.81      0.81      0.81        85
weighted avg       0.81      0.81      0.81        85



## XGBoostClassifier

In [62]:
from xgboost import XGBClassifier
pg={
    'n_estimators':[50,200,100,150],
    'max_depth':[4,5,6],
    'learning_rate':[0.5,0.8,0.3],
    }
gscv = GridSearchCV(XGBClassifier(random_state=7,subsample=0.8,colsample_bytree=0.8),
                    param_grid=pg,cv=2,scoring='average_precision',verbose=2)
gscv.fit(X_train,Y_train)

Fitting 2 folds for each of 36 candidates, totalling 72 fits
[CV] END ....learning_rate=0.5, max_depth=4, n_estimators=50; total time=   0.3s
[CV] END ....learning_rate=0.5, max_depth=4, n_estimators=50; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=4, n_estimators=200; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=4, n_estimators=200; total time=   0.2s
[CV] END ...learning_rate=0.5, max_depth=4, n_estimators=100; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=4, n_estimators=100; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=4, n_estimators=150; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=4, n_estimators=150; total time=   0.1s
[CV] END ....learning_rate=0.5, max_depth=5, n_estimators=50; total time=   0.1s
[CV] END ....learning_rate=0.5, max_depth=5, n_estimators=50; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=5, n_estimators=200; total time=   0.1s
[CV] END ...learning_rate=0.5, max_depth=5, n_es

GridSearchCV(cv=2,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=0.8, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'learning_rate': [0.5, 0.8, 0.3],
                         'max_depth': [4, 5, 6],
                         'n_estimators': [50, 200, 100, 150]},
             scoring='average_precision', verbose=2)

In [63]:
gscv.best_params_

{'learning_rate': 0.5, 'max_depth': 4, 'n_estimators': 100}

In [64]:
gscv.best_score_

np.float64(0.8703018107111158)

In [65]:
xgb = XGBClassifier(random_state=7,n_estimators=50,subsample=0.8,
                    max_depth=5,colsample_bytree=0.5,learning_rate=0.5)
xgb.fit(X_train,Y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.5, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, ...)

In [66]:
Y_pred = xgb.predict(X_test)
print(classification_report(Y_test,Y_pred))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80        46
           1       0.77      0.77      0.77        39

    accuracy                           0.79        85
   macro avg       0.79      0.79      0.79        85
weighted avg       0.79      0.79      0.79        85



## Cat Boost

In [70]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [76]:
from catboost import CatBoostClassifier
pg={
    'iterations':[50,100,200],
    'depth':[4,5,6,7],
    'learning_rate':[0.5,0.8,0.3]
}
gscv = GridSearchCV(CatBoostClassifier(random_state=7,cat_features=None,loss_function='MultiClass'),
                    param_grid=pg,cv=2,scoring='average_precision',verbose=2)
gscv.fit(X_train,Y_train)

Streaming output truncated to the last 5000 lines.
39:	learn: 0.0272279	total: 65.8ms	remaining: 263ms
40:	learn: 0.0263040	total: 68.6ms	remaining: 266ms
41:	learn: 0.0257275	total: 71.2ms	remaining: 268ms
42:	learn: 0.0250071	total: 73.9ms	remaining: 270ms
43:	learn: 0.0241894	total: 76.3ms	remaining: 271ms
44:	learn: 0.0235370	total: 79.4ms	remaining: 273ms
45:	learn: 0.0224562	total: 82.2ms	remaining: 275ms
46:	learn: 0.0217258	total: 85ms	remaining: 277ms
47:	learn: 0.0210324	total: 87.5ms	remaining: 277ms
48:	learn: 0.0205897	total: 90.1ms	remaining: 278ms
49:	learn: 0.0199115	total: 92.7ms	remaining: 278ms
50:	learn: 0.0195836	total: 95.3ms	remaining: 278ms
51:	learn: 0.0193790	total: 97ms	remaining: 276ms
52:	learn: 0.0188729	total: 98.5ms	remaining: 273ms
53:	learn: 0.0183665	total: 99.9ms	remaining: 270ms
54:	learn: 0.0179367	total: 101ms	remaining: 267ms
55:	learn: 0.0175093	total: 103ms	remaining: 265ms
56:	learn: 0.0171260	total: 105ms	remaining: 262ms
57:	learn: 0.0165836

GridSearchCV(cv=2,
             estimator=CatBoostClassifier(loss_function='MultiClass', random_state=7),
             param_grid={'depth': [4, 5, 6, 7], 'iterations': [50, 100, 200],
                         'learning_rate': [0.5, 0.8, 0.3]},
             scoring='average_precision', verbose=2)

In [77]:
gscv.best_params_

{'depth': 4, 'iterations': 200, 'learning_rate': 0.8}

In [79]:
gscv.best_score_

np.float64(0.8899213482755377)

## LightGBM

In [81]:
from lightgbm import LGBMClassifier
pg={
    'n_estimators':[5000,1000,7500],
    'max_depth':[4,5,6],
    'learning_rate':[0.5,0.8,0.3],
    }

gscv = GridSearchCV(LGBMClassifier(random_state=7,subsample=0.8,colsample_bytree=0.8),
                    param_grid=pg,cv=2,scoring='average_precision',verbose=2)
gscv.fit(X_train,Y_train)

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

GridSearchCV(cv=2,
             estimator=LGBMClassifier(colsample_bytree=0.8, random_state=7,
                                      subsample=0.8),
             param_grid={'learning_rate': [0.5, 0.8, 0.3],
                         'max_depth': [4, 5, 6],
                         'n_estimators': [5000, 1000, 7500]},
             scoring='average_precision', verbose=2)

In [82]:
gscv.best_params_

{'learning_rate': 0.8, 'max_depth': 4, 'n_estimators': 1000}

In [83]:
gscv.best_score_

np.float64(0.8563095101935975)